### Multiconductor & OpenDSS Load Flow Results

In [ ]:
def export_mc_loadflow_results(net, circuit_name):
    """Run multiconductor CCI power flow on *net* and export per-bus results to CSV.
    Returns a DataFrame of the results."""
    import csv
    import os
    import numpy as np
    import pandas as pd
    from multiconductor.pycci.cci_powerflow import run_pf

    results_csv = os.path.join(r"c:\Users\fgonzales\git\mc_opendss\networks\new\results", f"{circuit_name}_mc_loadflow_results.csv")

    # Run CCI power flow (populates net.res_bus, res_asymmetric_*, etc.)
    run_pf(net)

    PRECISION = 7
    bus_indices = net.bus.index.get_level_values(0).unique()

    # Phase number → letter mapping
    PHASE_LETTER = {1: 'A', 2: 'B', 3: 'C'}

    # ---- Aggregate element power per bus (total and per-phase) ----
    def _aggregate_power(element_name):
        """Return {bus_idx: {'total': [p, q], 'A': [p,q], 'B': [p,q], 'C': [p,q]}}."""
        elem_key = f"asymmetric_{element_name}"
        res_key  = f"res_asymmetric_{element_name}"
        result = {}
        if elem_key not in net or res_key not in net:
            return result
        elems = net[elem_key]
        res   = net[res_key]
        if elems.empty or res.empty:
            return result
        for idx in res.index:
            bus = int(elems.loc[idx, 'bus'])
            p = float(res.loc[idx, 'p_mw'])
            q = float(res.loc[idx, 'q_mvar'])
            from_phase = int(elems.loc[idx, 'from_phase'])
            letter = PHASE_LETTER.get(from_phase, None)
            if bus not in result:
                result[bus] = {'total': [0.0, 0.0], 'A': [0.0, 0.0], 'B': [0.0, 0.0], 'C': [0.0, 0.0]}
            result[bus]['total'][0] += p
            result[bus]['total'][1] += q
            if letter:
                result[bus][letter][0] += p
                result[bus][letter][1] += q
        return result

    load_power  = _aggregate_power('load')
    sgen_power  = _aggregate_power('sgen')
    shunt_power = _aggregate_power('shunt')

    rows = []
    for bus_idx in bus_indices:
        bus_data = net.bus.loc[bus_idx]
        if isinstance(bus_data, pd.DataFrame):
            name  = bus_data['name'].iloc[0]
            vn_kv = float(bus_data['vn_kv'].iloc[0])
        else:
            name  = bus_data['name']
            vn_kv = float(bus_data['vn_kv'])

        # Per-phase voltages from res_bus (MultiIndex: bus_idx, phase)
        v_pu  = {}
        v_ang = {}
        for phase in [1, 2, 3]:
            if (bus_idx, phase) in net.res_bus.index:
                vm = float(net.res_bus.loc[(bus_idx, phase), 'vm_pu'])
                va = float(net.res_bus.loc[(bus_idx, phase), 'va_degree'])
                if not np.isnan(vm):
                    v_pu[phase]  = round(vm, PRECISION)
                    v_ang[phase] = round(va, PRECISION)

        # Power at bus (total and per-phase)
        _empty = {'total': [0.0, 0.0], 'A': [0.0, 0.0], 'B': [0.0, 0.0], 'C': [0.0, 0.0]}
        lp = load_power.get(bus_idx, _empty)
        gp = sgen_power.get(bus_idx, _empty)
        sp = shunt_power.get(bus_idx, _empty)

        # Per-phase injection currents: I = conj(S / V)
        kv_ln = vn_kv / np.sqrt(3) if vn_kv > 0 else 0
        i_mag = {}
        i_ang_d = {}
        for phase in [1, 2, 3]:
            if phase in v_pu and v_pu[phase] > 0 and kv_ln > 0:
                if (bus_idx, phase) in net.res_bus.index:
                    p_mw   = float(net.res_bus.loc[(bus_idx, phase), 'p_mw'])
                    q_mvar = float(net.res_bus.loc[(bus_idx, phase), 'q_mvar'])
                    v_actual_v = v_pu[phase] * kv_ln * 1000.0
                    v_rad = np.deg2rad(v_ang[phase])
                    V_c = v_actual_v * np.exp(1j * v_rad)
                    if abs(V_c) > 0:
                        S = (p_mw * 1e6 + 1j * q_mvar * 1e6)
                        I_c = np.conj(S / V_c)
                        i_mag[phase]  = round(float(abs(I_c)), PRECISION)
                        i_ang_d[phase] = round(float(np.rad2deg(np.angle(I_c))), PRECISION)

        rows.append({
            'CKT_KEY': circuit_name,
            'NODE_ID': name,
            'BASE_VOLTAGE': round(vn_kv, PRECISION),
            'PGEN': round(gp['total'][0], PRECISION), 'QGEN': round(gp['total'][1], PRECISION),
            'PGEN_A': round(gp['A'][0], PRECISION), 'PGEN_B': round(gp['B'][0], PRECISION), 'PGEN_C': round(gp['C'][0], PRECISION),
            'QGEN_A': round(gp['A'][1], PRECISION), 'QGEN_B': round(gp['B'][1], PRECISION), 'QGEN_C': round(gp['C'][1], PRECISION),
            'PCAP': round(sp['total'][0], PRECISION), 'QCAP': round(sp['total'][1], PRECISION),
            'PLOAD': round(lp['total'][0], PRECISION), 'QLOAD': round(lp['total'][1], PRECISION),
            'PLOAD_A': round(lp['A'][0], PRECISION), 'PLOAD_B': round(lp['B'][0], PRECISION), 'PLOAD_C': round(lp['C'][0], PRECISION),
            'QLOAD_A': round(lp['A'][1], PRECISION), 'QLOAD_B': round(lp['B'][1], PRECISION), 'QLOAD_C': round(lp['C'][1], PRECISION),
            'VA': v_pu.get(1, None),  'VA_ANGLE': v_ang.get(1, None),
            'VB': v_pu.get(2, None),  'VB_ANGLE': v_ang.get(2, None),
            'VC': v_pu.get(3, None),  'VC_ANGLE': v_ang.get(3, None),
            'IA': i_mag.get(1, None), 'IA_ANGLE': i_ang_d.get(1, None),
            'IB': i_mag.get(2, None), 'IB_ANGLE': i_ang_d.get(2, None),
            'IC': i_mag.get(3, None), 'IC_ANGLE': i_ang_d.get(3, None),
        })

    df = pd.DataFrame(rows)
    df.to_csv(results_csv, index=False)
    print(f"Wrote {len(bus_indices)} buses to {results_csv}")
    return df

In [ ]:
def export_cyme_loadflow_results(circuit_name, sxst_file):

    # ---------------------------------------------------------------------------
    # Export load flow results to CSV
    # ---------------------------------------------------------------------------
    import csv
    import os
    import pandas as pd

    import sys
    sys.path.insert(0, r"C:\CYME\CYME")
    import cympy

    from datetime import datetime
    import csv
    import os
    import math
    import copy
    import re
    import decimal
    import logging


    # cympy.study.Close(AskForSave=False)
    print("Opening: ", sxst_file)
    cympy.study.Open(str(sxst_file))

    # ---------------------------------------------------------------------------
    # Load Flow — all active controls disabled
    # ---------------------------------------------------------------------------
    lf = cympy.sim.LoadFlow()

    # --- General solver settings ------------------------------------------------
    lf.SetValue(0.001,  'ParametersConfigurations[0].VoltageTolerance')
    lf.SetValue("VoltageDropUnbalanced", 'ParametersConfigurations[0].AnalysisMode')

    # Source / line modelling
    lf.SetValue(0, 'ParametersConfigurations[0].IncludeSourceImpedance')
    lf.SetValue(0, 'ParametersConfigurations[0].IncludeLineCharging')
    lf.SetValue(0, 'ParametersConfigurations[0].AssumeLineTransposition')

    # Temperature adjustment off
    lf.SetValue(0, 'ParametersConfigurations[0].TemperatureAdjustment.EnableTemperatureAdjustment')

    # Voltage-sensitivity load model
    lf.SetValue('Global', 'ParametersConfigurations[0].LoadFlowVoltageSensitivityLoadModel.Mode')
    lf.SetValue(0.0,     'ParametersConfigurations[0].LoadFlowVoltageSensitivityLoadModel.V')

    # --- Lock regulators / switched shunts (prevents tap & stage changes) -------
    lf.SetValue(1, 'ParametersConfigurations[0].LockMultiStageSwitchableShunts')
    lf.SetValue(1, 'ParametersConfigurations[0].LockSwitchedCapacitors')

    # --- Equipment status: disable ALL active controls --------------------------
    # Switched capacitor control modes — all OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.VoltageCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.CurrentCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.ReactiveCurrentCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.PowerFactorCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.TemperatureCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.TimeCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.KVARCapStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.PythonScriptCapStatus')

    # Fixed caps remain on (they have no switching logic)
    lf.SetValue(1, 'ParametersConfigurations[0].EquipmentStatusParameters.FixedCapStatus')

    # Multi-stage switchable shunts — OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.MultiStageSwitchableShuntStatus')

    # SVCs / VAR compensators — OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.SVCStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.VARCompensatorStatus')

    # Motors — OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.SynchronousMotorStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.InductionMotorStatus')

    # Generators — keep synchronous & ECG energised but all others OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.GeneratorStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.SynchronousGeneratorStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.InductionGeneratorStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.ElectronicallyCoupledGeneratorStatus')

    # Renewables / storage — OFF
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.WecsStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.BESSStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.SofcStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.PhotovoltaicStatus')
    lf.SetValue(0, 'ParametersConfigurations[0].EquipmentStatusParameters.MicroTurbineStatus')

    # --- Run load flow -----------------------------------------------------------
    lf.Run()
    print("Load flow complete (all active controls disabled)")

    PRECISION = 7
    results_csv = os.path.join(r"c:\Users\fgonzales\git\mc_opendss\networks\new\results", f"{circuit_name}_cyme_loadflow_results.csv")

    nodes = cympy.study.ListNodes(cympy.enums.NodeType.All)

    rows = []
    for node in nodes:
        nid = node.ID
        rows.append({
            'CKT_KEY': circuit_name,
            'NODE_ID': nid,
            'BASE_VOLTAGE': cympy.study.QueryInfoNode('KVLLBase',   nid, PRECISION),
            'PGEN':         cympy.study.QueryInfoNode('GenMW',      nid, PRECISION),
            'QGEN':         cympy.study.QueryInfoNode('GenMVAR',    nid, PRECISION),
            'PGEN_A':       None,  # CYME QueryInfoNode has no per-phase gen keywords
            'PGEN_B':       None,
            'PGEN_C':       None,
            'QGEN_A':       None,
            'QGEN_B':       None,
            'QGEN_C':       None,
            'PCAP':         cympy.study.QueryInfoNode('CapMW',      nid, PRECISION),
            'QCAP':         cympy.study.QueryInfoNode('CapMVAR',    nid, PRECISION),
            'PLOAD':        cympy.study.QueryInfoNode('LoadMW',     nid, PRECISION),
            'QLOAD':        cympy.study.QueryInfoNode('LoadMVAR',   nid, PRECISION),
            'PLOAD_A':      None,  # CYME QueryInfoNode has no per-phase load keywords
            'PLOAD_B':      None,
            'PLOAD_C':      None,
            'QLOAD_A':      None,
            'QLOAD_B':      None,
            'QLOAD_C':      None,
            'VA':           cympy.study.QueryInfoNode('VpuA',       nid, PRECISION),
            'VA_ANGLE':     cympy.study.QueryInfoNode('PH-AngleA',  nid, PRECISION),
            'VB':           cympy.study.QueryInfoNode('VpuB',       nid, PRECISION),
            'VB_ANGLE':     cympy.study.QueryInfoNode('PH-AngleB',  nid, PRECISION),
            'VC':           cympy.study.QueryInfoNode('VpuC',       nid, PRECISION),
            'VC_ANGLE':     cympy.study.QueryInfoNode('PH-AngleC',  nid, PRECISION),
            'IA':           cympy.study.QueryInfoNode('IA',         nid, PRECISION),
            'IA_ANGLE':     cympy.study.QueryInfoNode('IAngleA',    nid, PRECISION),
            'IB':           cympy.study.QueryInfoNode('IB',         nid, PRECISION),
            'IB_ANGLE':     cympy.study.QueryInfoNode('IAngleB',    nid, PRECISION),
            'IC':           cympy.study.QueryInfoNode('IC',         nid, PRECISION),
            'IC_ANGLE':     cympy.study.QueryInfoNode('IAngleC',    nid, PRECISION),
        })

    df = pd.DataFrame(rows)
    df.to_csv(results_csv, index=False)
    print(f"Wrote {len(nodes)} nodes to {results_csv}")
    return df

In [ ]:
def export_dss_loadflow_results(circuit_name, dss_file):
    """Run OpenDSS power flow on a .dss file and export per-bus results to CSV.
    Returns a DataFrame of the results."""
    import csv
    import os
    import math
    import numpy as np
    import pandas as pd

    from opendss.pf.powerflow import run_pf as dss_run_pf

    results_csv = os.path.join(r"c:\Users\fgonzales\git\mc_opendss\networks\new\results", f"{circuit_name}_dss_loadflow_results.csv")

    # Run OpenDSS power flow
    pf = dss_run_pf(str(dss_file))
    d = pf['dss']

    if not pf['converged']:
        print(f"WARNING: DSS power flow did not converge for {circuit_name}")

    PRECISION = 7

    # ---- Build per-bus power maps from element iteration ----
    def _collect_element_powers(element_class, element_prefix):
        """Iterate DSS elements, return {bus_name_lower: {phase: [p_kw, q_kvar]}}."""
        power_map = {}
        elem_api = getattr(d, element_class, None)
        if elem_api is None:
            return power_map
        try:
            i = elem_api.first()
        except Exception:
            return power_map
        while i != 0:
            ename = elem_api.name
            d.circuit.set_active_element(f"{element_prefix}.{ename}")
            bus_names = d.cktelement.bus_names
            powers = d.cktelement.powers          # [P1, Q1, P2, Q2, ...]
            n = d.cktelement.num_conductors
            if bus_names:
                parts = bus_names[0].split('.')
                bname = parts[0].lower()
                phases = [int(p) for p in parts[1:]] if len(parts) > 1 else list(range(1, n + 1))
                if bname not in power_map:
                    power_map[bname] = {}
                for j, ph in enumerate(phases):
                    if j * 2 + 1 < len(powers):
                        if ph not in power_map[bname]:
                            power_map[bname][ph] = [0.0, 0.0]
                        power_map[bname][ph][0] += powers[j * 2]
                        power_map[bname][ph][1] += powers[j * 2 + 1]
            i = elem_api.next()
        return power_map

    load_map = _collect_element_powers('loads', 'Load')
    gen_map  = _collect_element_powers('generators', 'Generator')
    cap_map  = _collect_element_powers('capacitors', 'Capacitor')

    # Phase number → letter mapping
    PHASE_LETTER = {1: 'A', 2: 'B', 3: 'C'}

    num_buses = d.circuit.num_buses
    rows = []

    for i in range(num_buses):
        d.circuit.set_active_bus_i(i)
        name   = d.bus.name
        kv_base = d.bus.kv_base              # L-N kV
        nodes  = list(d.bus.nodes)            # e.g. [1, 2, 3]
        va_pu  = d.bus.vmag_angle_pu          # [mag1, ang1, mag2, ang2, ...]

        # Per-phase voltage magnitudes (pu) and angles (deg)
        v_pu  = {}
        v_ang = {}
        for j, node in enumerate(nodes):
            if j * 2 + 1 < len(va_pu):
                v_pu[node]  = round(va_pu[j * 2],     PRECISION)
                v_ang[node] = round(va_pu[j * 2 + 1], PRECISION)

        # Total per-bus power (sum across phases), kW → MW
        name_lower = name.lower()
        lp = load_map.get(name_lower, {})
        gp = gen_map.get(name_lower, {})
        cp = cap_map.get(name_lower, {})

        pload = round(sum(v[0] for v in lp.values()) / 1000.0, PRECISION)
        qload = round(sum(v[1] for v in lp.values()) / 1000.0, PRECISION)
        pgen  = round(sum(v[0] for v in gp.values()) / 1000.0, PRECISION)
        qgen  = round(sum(v[1] for v in gp.values()) / 1000.0, PRECISION)
        pcap  = round(sum(v[0] for v in cp.values()) / 1000.0, PRECISION)
        qcap  = round(sum(v[1] for v in cp.values()) / 1000.0, PRECISION)

        # Per-phase load and gen power (kW → MW, kVAR → MVAR)
        per_phase_load = {}
        per_phase_gen = {}
        for ph, letter in PHASE_LETTER.items():
            lph = lp.get(ph, [0.0, 0.0])
            gph = gp.get(ph, [0.0, 0.0])
            per_phase_load[f'PLOAD_{letter}'] = round(lph[0] / 1000.0, PRECISION)
            per_phase_load[f'QLOAD_{letter}'] = round(lph[1] / 1000.0, PRECISION)
            per_phase_gen[f'PGEN_{letter}']   = round(gph[0] / 1000.0, PRECISION)
            per_phase_gen[f'QGEN_{letter}']   = round(gph[1] / 1000.0, PRECISION)

        # Per-phase injection currents: I = conj(S / V)
        i_mag = {}
        i_ang_d = {}
        for phase in [1, 2, 3]:
            if phase in v_pu and v_pu[phase] > 0 and kv_base > 0:
                v_actual_v = v_pu[phase] * kv_base * 1000.0
                v_rad = np.deg2rad(v_ang[phase])
                V_c = v_actual_v * np.exp(1j * v_rad)
                p_w  = ((gp.get(phase, [0, 0])[0] - lp.get(phase, [0, 0])[0]
                         + cp.get(phase, [0, 0])[0]) * 1000.0)
                q_va = ((gp.get(phase, [0, 0])[1] - lp.get(phase, [0, 0])[1]
                         + cp.get(phase, [0, 0])[1]) * 1000.0)
                if abs(V_c) > 0:
                    I_c = np.conj((p_w + 1j * q_va) / V_c)
                    i_mag[phase]  = round(float(abs(I_c)), PRECISION)
                    i_ang_d[phase] = round(float(np.rad2deg(np.angle(I_c))), PRECISION)

        kv_ll = round(kv_base * math.sqrt(3), PRECISION) if kv_base > 0 else 0

        row = {
            'CKT_KEY': circuit_name,
            'NODE_ID': name,
            'BASE_VOLTAGE': kv_ll,
            'PGEN': pgen, 'QGEN': qgen,
            'PGEN_A': per_phase_gen['PGEN_A'], 'PGEN_B': per_phase_gen['PGEN_B'], 'PGEN_C': per_phase_gen['PGEN_C'],
            'QGEN_A': per_phase_gen['QGEN_A'], 'QGEN_B': per_phase_gen['QGEN_B'], 'QGEN_C': per_phase_gen['QGEN_C'],
            'PCAP': pcap, 'QCAP': qcap,
            'PLOAD': pload, 'QLOAD': qload,
            'PLOAD_A': per_phase_load['PLOAD_A'], 'PLOAD_B': per_phase_load['PLOAD_B'], 'PLOAD_C': per_phase_load['PLOAD_C'],
            'QLOAD_A': per_phase_load['QLOAD_A'], 'QLOAD_B': per_phase_load['QLOAD_B'], 'QLOAD_C': per_phase_load['QLOAD_C'],
            'VA': v_pu.get(1, None),  'VA_ANGLE': v_ang.get(1, None),
            'VB': v_pu.get(2, None),  'VB_ANGLE': v_ang.get(2, None),
            'VC': v_pu.get(3, None),  'VC_ANGLE': v_ang.get(3, None),
            'IA': i_mag.get(1, None), 'IA_ANGLE': i_ang_d.get(1, None),
            'IB': i_mag.get(2, None), 'IB_ANGLE': i_ang_d.get(2, None),
            'IC': i_mag.get(3, None), 'IC_ANGLE': i_ang_d.get(3, None),
        }
        rows.append(row)

    df = pd.DataFrame(rows)
    df.to_csv(results_csv, index=False)
    print(f"Wrote {num_buses} buses to {results_csv}")
    return df

In [ ]:
import pickle, time
from pathlib import Path
from IPython.display import display, HTML

mc_dir   = Path(r"c:\Users\fgonzales\git\mc_opendss\networks\new\mc")
dss_dir  = Path(r"c:\Users\fgonzales\git\mc_opendss\networks\new\dss")
cyme_dir = Path(r"c:\Users\fgonzales\git\mc_opendss\networks\new\cyme")

pkl_files = sorted(mc_dir.glob("*.pkl"))
total = len(pkl_files)

all_results = {}
progress = display(HTML("Starting…"), display_id=True)
t_start = time.time()
n_ok, n_fail = 0, 0

for i, pkl_path in enumerate(pkl_files, 1):
    circuit_key = pkl_path.stem
    t0 = time.time()
    try:
        with open(pkl_path, "rb") as fh:
            net = pickle.load(fh)

        df_mc  = export_mc_loadflow_results(net, circuit_key)
        dss_file = dss_dir / f"{circuit_key}.dss"
        df_dss = export_dss_loadflow_results(circuit_key, str(dss_file))

        df_cyme = None
        cyme_file = cyme_dir / f"{circuit_key}.sxst"
        if cyme_file.exists():
            df_cyme = export_cyme_loadflow_results(circuit_key, cyme_file)

        all_results[circuit_key] = {'mc': df_mc, 'dss': df_dss, 'cyme': df_cyme, 'net': net}
        n_ok += 1
        elapsed = time.time() - t0
        avg = (time.time() - t_start) / i
        eta_m, eta_s = divmod(int(avg * (total - i)), 60)
        pct = i / total * 100
        filled = int(30 * i // total)
        bar = "█" * filled + "░" * (30 - filled)
        cyme_tag = " +CYME" if df_cyme is not None else ""
        progress.update(HTML(
            f"<b>[{i}/{total}]</b> {bar} {pct:.0f}% │ "
            f"<code>{circuit_key}</code> ({elapsed:.1f}s{cyme_tag}) │ "
            f"ETA {eta_m}m{eta_s:02d}s"
        ))
    except Exception as e:
        n_fail += 1
        progress.update(HTML(
            f"<b>[{i}/{total}]</b> <code>{circuit_key}</code> ✗ {e}"
        ))

total_time = time.time() - t_start
progress.update(HTML(
    f"<b>Done in {total_time:.1f}s — {n_ok} OK, {n_fail} failed, "
    f"{len(all_results)} circuits loaded.</b>"
))
print(f"{len(all_results)} circuits in all_results")

In [ ]:
from sce.load_allocation_utils import get_asymmetric_load_by_phase
from sce.load_allocation import SCELoadAllocationMeasurement

la = SCELoadAllocationMeasurement()



In [ ]:
from sce.newton_rules_engine import get_pf_rule_mismatches, new_pf_field_mapping
from sce import nr_iteration_scrubber as scrubber
from multiconductor.pycci.cci_powerflow import run_pf
import pickle

pkl_file = f"c:\\Users\\fgonzales\\git\\mc_opendss\\networks\\new\\mc\\CKT_114_16955.pkl"
with open(pkl_file, 'rb') as fh:
    net = pickle.load(fh)
#mismatch_df = get_pf_rule_mismatches(net, new_pf_field_mapping)
#mismatch_df
net.ext_grid_sequence['r_ohm'] = 1e-9
net.ext_grid_sequence['x_ohm'] = 1e-9

#la.cc_load_allocation(net, 1.5)

run_pf(net, run_control=False)


In [ ]:
net.res_bus

In [ ]:
ld = get_asymmetric_load_by_phase(net)
ld

In [ ]:
p_mw_A = ld['p_mw_A'].sum()
p_mw_B = ld['p_mw_B'].sum()
p_mw_C = ld['p_mw_C'].sum()

print(f"Total load: {p_mw_A + p_mw_B + p_mw_C:.2f} MW (A: {p_mw_A:.2f} MW, B: {p_mw_B:.2f} MW, C: {p_mw_C:.2f} MW)")

In [ ]:
import pickle
from pathlib import Path
import pandas as pd
import opendss.tools.dss_converter as odc
from multiconductor.pycci.cci_powerflow import run_pf


CKT_4053_04314 = 'CKT_4053_04314'
CKT_114_16955 = 'CKT_114_16955'
CKT_4799_01085 = 'CKT_4799_01085'

validation_nets = ['CKT_2292_10360']

all_results = {} 

for circuit_key in validation_nets:
    print(f"Processing {circuit_key}...")
    #                  C:\Users\fgonzales\git\mc_opendss\networks\new\mc
    pkl_file = Path(f"c:\\Users\\fgonzales\\git\\mc-opendss\\networks\\new\\mc\\{circuit_key}.pkl")
    dss_file = Path(rf"c:\Users\fgonzales\git\mc-opendss\networks\new\dss\{circuit_key}.dss")
    cyme_file = Path(rf"c:\Users\fgonzales\git\mc-opendss\networks\new\cyme\{circuit_key}.sxst")

    with open(pkl_file, 'rb') as fh:
        net = pickle.load(fh)

    df_mc = export_mc_loadflow_results(net, circuit_key)    
    df_dss = export_dss_loadflow_results(circuit_key, dss_file)
    df_cyme = None
    if cyme_file.exists():
        df_cyme = export_cyme_loadflow_results(circuit_key, cyme_file)

    all_results[circuit_key] = {'mc': df_mc, 'dss': df_dss, 'cyme': df_cyme, 'net': net}

print("All circuits processed.")

In [ ]:
import plotly.graph_objects as go
import pandas as pd
import numpy as np

NUMERIC_FIELDS = [
    'BASE_VOLTAGE',
    'PGEN', 'QGEN',
    'PGEN_A', 'PGEN_B', 'PGEN_C',
    'QGEN_A', 'QGEN_B', 'QGEN_C',
    'PCAP', 'QCAP',
    'PLOAD', 'QLOAD',
    'PLOAD_A', 'PLOAD_B', 'PLOAD_C',
    'QLOAD_A', 'QLOAD_B', 'QLOAD_C',
    'VA', 'VA_ANGLE', 'VB', 'VB_ANGLE', 'VC', 'VC_ANGLE',
    'IA', 'IA_ANGLE', 'IB', 'IB_ANGLE', 'IC', 'IC_ANGLE',
]

SOURCE_COLORS  = {'mc': '#1f77b4', 'dss': '#ff7f0e', 'cyme': '#2ca02c'}
SOURCE_SYMBOLS = {'mc': 'circle',  'dss': 'diamond', 'cyme': 'square'}
SOURCES = ['mc', 'dss', 'cyme']
circuit_keys = list(all_results.keys())

def _get_trace_data(ckt, field, label):
    """Return (x, y, text) arrays for one source trace."""
    df = all_results.get(ckt, {}).get(label)
    if df is not None and not df.empty and field in df.columns:
        col = pd.to_numeric(df[field], errors='coerce')
        mask = col.notna()
        return np.arange(mask.sum()), col[mask].values, df.loc[mask, 'NODE_ID'].values
    return np.array([]), np.array([]), np.array([])

# --- Create figure with 3 traces (MC, DSS, CYME) ---
default_ckt = circuit_keys[0]
default_field = 'VA'

fig = go.Figure()
for label in SOURCES:
    x, y, text = _get_trace_data(default_ckt, default_field, label)
    fig.add_trace(go.Scatter(
        x=x, y=y, text=text,
        mode='markers',
        name=label.upper(),
        marker=dict(color=SOURCE_COLORS[label], symbol=SOURCE_SYMBOLS[label], size=6, opacity=0.7),
        hovertemplate='%{text}<br>' + default_field + ': %{y}<extra></extra>',
    ))

# --- Circuit dropdown ---
circuit_buttons = []
for ckt in circuit_keys:
    xd, yd, td = [], [], []
    for label in SOURCES:
        x, y, text = _get_trace_data(ckt, default_field, label)
        xd.append(x); yd.append(y); td.append(text)
    ht = ['%{text}<br>' + default_field + ': %{y}<extra></extra>'] * len(SOURCES)
    circuit_buttons.append(dict(
        label=ckt, method='update',
        args=[{'x': xd, 'y': yd, 'text': td, 'hovertemplate': ht},
              {'title': f'{ckt} — {default_field}', 'yaxis.title': default_field}],
    ))

# --- Field dropdown ---
field_buttons = []
for field in NUMERIC_FIELDS:
    xd, yd, td = [], [], []
    for label in SOURCES:
        x, y, text = _get_trace_data(default_ckt, field, label)
        xd.append(x); yd.append(y); td.append(text)
    ht = ['%{text}<br>' + field + ': %{y}<extra></extra>'] * len(SOURCES)
    field_buttons.append(dict(
        label=field, method='update',
        args=[{'x': xd, 'y': yd, 'text': td, 'hovertemplate': ht},
              {'title': f'{default_ckt} — {field}', 'yaxis.title': field}],
    ))

# --- Source toggle buttons ---
source_buttons = [
    dict(label='All',  method='restyle', args=[{'visible': [True, True, True]}]),
    dict(label='MC',   method='restyle', args=[{'visible': [True, 'legendonly', 'legendonly']}]),
    dict(label='DSS',  method='restyle', args=[{'visible': ['legendonly', True, 'legendonly']}]),
    dict(label='CYME', method='restyle', args=[{'visible': ['legendonly', 'legendonly', True]}]),
]

fig.update_layout(
    template='plotly_dark',
    title=f'{default_ckt} — {default_field}',
    xaxis_title='Node index',
    yaxis_title=default_field,
    height=550,
    legend_title='Source',
    updatemenus=[
        dict(buttons=circuit_buttons, direction='down', showactive=True,
             x=0.0, xanchor='left', y=1.18, yanchor='top'),
        dict(buttons=field_buttons, direction='down', showactive=True,
             x=0.28, xanchor='left', y=1.18, yanchor='top'),
        dict(buttons=source_buttons, type='buttons', direction='right',
             showactive=True, active=0,
             x=0.55, xanchor='left', y=1.18, yanchor='top'),
    ],
    annotations=[
        dict(text='Circuit:', x=0.0, xref='paper', xanchor='right',
             y=1.15, yref='paper', showarrow=False, xshift=-5),
        dict(text='Field:', x=0.28, xref='paper', xanchor='right',
             y=1.15, yref='paper', showarrow=False, xshift=-5),
        dict(text='Sources:', x=0.55, xref='paper', xanchor='right',
             y=1.15, yref='paper', showarrow=False, xshift=-5),
    ],
)
fig.show()

In [ ]:
import pandas as pd
import numpy as np
import os

COMPARE_FIELDS = [
    'BASE_VOLTAGE',
    'PGEN', 'QGEN',
    'PGEN_A', 'PGEN_B', 'PGEN_C',
    'QGEN_A', 'QGEN_B', 'QGEN_C',
    'PCAP', 'QCAP',
    'PLOAD', 'QLOAD',
    'PLOAD_A', 'PLOAD_B', 'PLOAD_C',
    'QLOAD_A', 'QLOAD_B', 'QLOAD_C',
    'VA', 'VA_ANGLE', 'VB', 'VB_ANGLE', 'VC', 'VC_ANGLE',
    'IA', 'IA_ANGLE', 'IB', 'IB_ANGLE', 'IC', 'IC_ANGLE',
]

def _build_element_map(net):
    """Return {bus_name: comma-separated element types} from the net object."""
    bus_elements = {}  # bus_idx -> set of element type strings

    element_tables = [
        ('asymmetric_load',  'bus', 'load'),
        ('asymmetric_sgen',  'bus', 'sgen'),
        ('asymmetric_shunt', 'bus', 'shunt'),
    ]
    for table_name, bus_col, label in element_tables:
        if table_name in net and not net[table_name].empty:
            for bus_idx in net[table_name][bus_col].unique():
                bus_elements.setdefault(int(bus_idx), set()).add(label)

    # 2-winding transformers (trafo or trafo1ph)
    if 'trafo' in net and not net['trafo'].empty:
        for _, row in net['trafo'].iterrows():
            for col in ['hv_bus', 'lv_bus']:
                if col in row.index:
                    bus_elements.setdefault(int(row[col]), set()).add('trafo')

    # trafo1ph: bus is stored in the MultiIndex, not as a column
    if 'trafo1ph' in net and not net['trafo1ph'].empty:
        for bus_idx in net['trafo1ph'].index.get_level_values('bus').unique():
            bus_elements.setdefault(int(bus_idx), set()).add('trafo')

    # 3-winding transformers
    if 'trafo3w' in net and not net['trafo3w'].empty:
        for _, row in net['trafo3w'].iterrows():
            for col in ['hv_bus', 'mv_bus', 'lv_bus']:
                if col in row.index:
                    bus_elements.setdefault(int(row[col]), set()).add('trafo3w')

    # ext_grid (source/gen) — check both ext_grid and ext_grid_sequence
    for eg_name in ['ext_grid', 'ext_grid_sequence']:
        if eg_name in net and not net[eg_name].empty:
            if 'bus' in net[eg_name].columns:
                for bus_idx in net[eg_name]['bus'].unique():
                    bus_elements.setdefault(int(bus_idx), set()).add('ext_grid')
            elif 'bus' in net[eg_name].index.names:
                for bus_idx in net[eg_name].index.get_level_values('bus').unique():
                    bus_elements.setdefault(int(bus_idx), set()).add('ext_grid')

    # Map bus index → bus name
    name_map = {}
    for bus_idx in net.bus.index.get_level_values(0).unique():
        bus_data = net.bus.loc[bus_idx]
        name = bus_data['name'].iloc[0] if isinstance(bus_data, pd.DataFrame) else bus_data['name']
        name_map[int(bus_idx)] = name

    # Build {bus_name: element_string}
    result = {}
    for bus_idx, elem_set in bus_elements.items():
        if bus_idx in name_map:
            result[name_map[bus_idx]] = ', '.join(sorted(elem_set))
    return result

def compare_loadflow_results(df_mc, df_dss, circuit_name, df_cyme=None, net=None, output_dir=r"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\validation"):
    """Compare MC, DSS, and optionally CYME load flow results side-by-side with % difference.

    Joins on NODE_ID, then for each numeric field produces columns:
        <field>_MC, <field>_DSS, <field>_%DIFF_DSS
    and if df_cyme is provided:
        <field>_CYME, <field>_%DIFF_CYME
    where %DIFF = (MC - other) / MC * 100.  When MC == 0 the diff is NaN.

    Returns the merged DataFrame and writes it to CSV.
    """
    mc = df_mc[['NODE_ID'] + COMPARE_FIELDS].copy()
    dss = df_dss[['NODE_ID'] + COMPARE_FIELDS].copy()

    for col in COMPARE_FIELDS:
        mc[col] = pd.to_numeric(mc[col], errors='coerce')
        dss[col] = pd.to_numeric(dss[col], errors='coerce')

    # Rename columns before merge to avoid suffix collisions
    mc = mc.rename(columns={c: f'{c}_MC' for c in COMPARE_FIELDS})
    dss = dss.rename(columns={c: f'{c}_DSS' for c in COMPARE_FIELDS})

    merged = mc.merge(dss, on='NODE_ID', how='outer')

    # Compute MC vs DSS % diff
    for col in COMPARE_FIELDS:
        mc_col = f'{col}_MC'
        dss_col = f'{col}_DSS'
        diff_col = f'{col}_%DIFF_DSS'
        merged[diff_col] = np.where(
            merged[mc_col] != 0,
            (merged[mc_col] - merged[dss_col]) / merged[mc_col] * 100,
            np.nan,
        )

    # Merge CYME if provided
    if df_cyme is not None:
        cyme = df_cyme[['NODE_ID'] + COMPARE_FIELDS].copy()
        cyme['NODE_ID'] = cyme['NODE_ID'].astype(str)
        merged['NODE_ID'] = merged['NODE_ID'].astype(str)
        for col in COMPARE_FIELDS:
            cyme[col] = pd.to_numeric(cyme[col], errors='coerce')
        cyme = cyme.rename(columns={c: f'{c}_CYME' for c in COMPARE_FIELDS})
        merged = merged.merge(cyme, on='NODE_ID', how='outer')

        for col in COMPARE_FIELDS:
            mc_col = f'{col}_MC'
            cyme_col = f'{col}_CYME'
            diff_col = f'{col}_%DIFF_CYME'
            merged[diff_col] = np.where(
                merged[mc_col] != 0,
                (merged[mc_col] - merged[cyme_col]) / merged[mc_col] * 100,
                np.nan,
            )

    # Add circuit key
    merged.insert(0, 'CKT_KEY', circuit_name)

    # Add element type column from net
    if net is not None:
        elem_map = _build_element_map(net)
        merged.insert(2, 'ELEMENT', merged['NODE_ID'].map(elem_map).fillna(''))
    else:
        merged.insert(2, 'ELEMENT', '')

    # Reorder columns: CKT_KEY, NODE_ID, ELEMENT, then groups per field
    ordered_cols = ['CKT_KEY', 'NODE_ID', 'ELEMENT']
    for col in COMPARE_FIELDS:
        ordered_cols += [f'{col}_MC', f'{col}_DSS', f'{col}_%DIFF_DSS']
        if df_cyme is not None:
            ordered_cols += [f'{col}_CYME', f'{col}_%DIFF_CYME']
    merged = merged[ordered_cols]

    out_csv = os.path.join(output_dir, f"{circuit_name}_mc_vs_dss_comparison.csv")
    merged.to_csv(out_csv, index=False)
    print(f"Wrote {len(merged)} rows to {out_csv}")
    return merged

# Run comparison for all circuits
comparison_results = {}
df = pd.read_csv(rf"c:\Users\fgonzales\git\mc-0.0.1.15\scripts\encoded.csv")
for row in df.itertuples():
    ckt = row.CIRCUIT_KEY
    mc_csv = rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\validation\{ckt}_mc_loadflow_results.csv"
    dss_csv = rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\validation\{ckt}_dss_loadflow_results.csv"
    cyme_csv = rf"c:\Users\fgonzales\git\mc-0.0.1.15\networks_final\validation\{ckt}_cyme_loadflow_results.csv"

    if not os.path.isfile(mc_csv) or not os.path.isfile(dss_csv):
        print(f"Skipping {ckt} — missing MC or DSS CSV(s)")
        continue

    df_mc = pd.read_csv(mc_csv)
    df_dss = pd.read_csv(dss_csv)
    df_cyme = pd.read_csv(cyme_csv) if os.path.isfile(cyme_csv) else None

    print(f"Comparing {ckt}..." + (" (with CYME)" if df_cyme is not None else ""))
    comparison_results[ckt] = compare_loadflow_results(df_mc, df_dss, ckt, df_cyme=df_cyme)

comparison_results[list(comparison_results.keys())[0]].head(10)

In [ ]:
all_df = comparison_results[list(comparison_results.keys())[0]]
all_df

In [ ]:
all_df = comparison_results[list(comparison_results.keys())[1]]

In [ ]:
cols = ["CKT_KEY", "NODE_ID",
    "VA_MC", "VA_DSS", "VA_CYME",
    "VB_MC", "VB_DSS", "VB_CYME",
    "VC_MC", "VC_DSS", "VC_CYME"
]

new_df = all_df[cols].copy()

In [ ]:

new_df

In [ ]:
numeric_cols = new_df.select_dtypes(include='number').columns.tolist()
summary = new_df.groupby('CKT_KEY')[numeric_cols].mean()
summary

In [ ]:
import pickle, time, sys, os
from pathlib import Path
import pandas as pd
import numpy as np
import tempfile
import opendss.tools.dss_converter as odc
from multiconductor.pycci.cci_powerflow import run_pf as mc_run_pf

# ── CYME setup ──────────────────────────────────────────────────────────────
cyme_python_path = os.environ.get("CYME_PYTHON_PATH", r"C:\CYME\CYME")
if cyme_python_path not in sys.path:
    sys.path.insert(0, cyme_python_path)
import cympy
from cyme.pf.powerflow import run_pf as cyme_run_pf

PRECISION = 7

def _safe_float(val):
    try:
        v = float(val)
        return v if v > 0 else None
    except (TypeError, ValueError):
        return None

# ── Iterate mc_xfmr networks ───────────────────────────────────────────────
# Use absolute paths so DSS working-directory changes don't break file access
src_dir = Path(r"c:\Users\fgonzales\git\mc_opendss\networks\mc_xfmr")
cyme_dir = Path(r"c:\Users\fgonzales\git\mc_opendss\networks\cyme")
pkl_files = sorted(src_dir.glob("*.pkl"))
total = len(pkl_files)

rows = []
t_start = time.time()

for i, pkl_path in enumerate(pkl_files, 1):
    circuit_key = pkl_path.stem
    t0 = time.time()

    try:
        with open(pkl_path, "rb") as fh:
            net = pickle.load(fh)

        # ── MC power flow ───────────────────────────────────────────────
        mc_run_pf(net)
        mc_va = pd.to_numeric(
            net.res_bus.loc[net.res_bus.index.get_level_values(1) == 1, "vm_pu"],
            errors="coerce",
        ).dropna()
        mc_vb = pd.to_numeric(
            net.res_bus.loc[net.res_bus.index.get_level_values(1) == 2, "vm_pu"],
            errors="coerce",
        ).dropna()
        mc_vc = pd.to_numeric(
            net.res_bus.loc[net.res_bus.index.get_level_values(1) == 3, "vm_pu"],
            errors="coerce",
        ).dropna()

        # ── DSS power flow ──────────────────────────────────────────────
        dss_file = Path(tempfile.gettempdir()) / f"{circuit_key}.dss"
        odc.mc_net_to_opendss(net, str(dss_file))
        df_dss = export_dss_loadflow_results(circuit_key, dss_file)

        dss_va = pd.to_numeric(df_dss["VA"], errors="coerce").dropna()
        dss_vb = pd.to_numeric(df_dss["VB"], errors="coerce").dropna()
        dss_vc = pd.to_numeric(df_dss["VC"], errors="coerce").dropna()

        # ── CYME power flow ─────────────────────────────────────────────
        cyme_va_mean, cyme_vb_mean, cyme_vc_mean = np.nan, np.nan, np.nan
        sxst_path = cyme_dir / f"{circuit_key}.sxst"
        if sxst_path.exists():
            cyme_result = cyme_run_pf(str(sxst_path), close_on_finish=False)

            cyme_va_vals, cyme_vb_vals, cyme_vc_vals = [], [], []
            nodes = cympy.study.ListNodes(cympy.enums.NodeType.All)
            for node in nodes:
                nid = node.ID
                va = _safe_float(cympy.study.QueryInfoNode("VpuA", nid, PRECISION))
                vb = _safe_float(cympy.study.QueryInfoNode("VpuB", nid, PRECISION))
                vc = _safe_float(cympy.study.QueryInfoNode("VpuC", nid, PRECISION))
                if va is not None:
                    cyme_va_vals.append(va)
                if vb is not None:
                    cyme_vb_vals.append(vb)
                if vc is not None:
                    cyme_vc_vals.append(vc)

            cympy.study.Close(AskForSave=False)
            cyme_va_mean = np.mean(cyme_va_vals) if cyme_va_vals else np.nan
            cyme_vb_mean = np.mean(cyme_vb_vals) if cyme_vb_vals else np.nan
            cyme_vc_mean = np.mean(cyme_vc_vals) if cyme_vc_vals else np.nan

        rows.append({
            "CKT_KEY": circuit_key,
            "MC_VA_PU": mc_va.mean(),
            "MC_VB_PU": mc_vb.mean(),
            "MC_VC_PU": mc_vc.mean(),
            "DSS_VA_PU": dss_va.mean(),
            "DSS_VB_PU": dss_vb.mean(),
            "DSS_VC_PU": dss_vc.mean(),
            "CYME_VA_PU": cyme_va_mean,
            "CYME_VB_PU": cyme_vb_mean,
            "CYME_VC_PU": cyme_vc_mean,
        })

        elapsed = time.time() - t0
        total_elapsed = time.time() - t_start
        avg = total_elapsed / i
        eta = avg * (total - i)
        print(f"[{i}/{total}] {circuit_key} ({elapsed:.1f}s, ETA {eta:.0f}s)")

    except Exception as e:
        print(f"[{i}/{total}] {circuit_key} FAILED: {e}")
        try:
            cympy.study.Close(AskForSave=False)
        except Exception:
            pass

df_pf_comparison = pd.DataFrame(rows)
print(f"\nDone in {time.time() - t_start:.1f}s — {len(df_pf_comparison)} circuits.")
df_pf_comparison